# Patient-Specific Seizure Detection — LORO Evaluation

Fixed-hyperparameter pipeline (no SVM tuning). For each patient:
LORO over seizure records, per-fold operating-point calibration on
out-of-fold training scores, SzCORE event-level scoring.

**Set your hyperparameters once in the CONFIG cell.** AP / ROC-AUC are
reported per fold as leakage-free diagnostics only — they select nothing.

## Imports

In [ ]:
pip install timescoring

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
# single-thread BLAS so the outer joblib parallelism over folds is clean
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import hashlib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (LeaveOneGroupOut, GroupKFold,
                                     StratifiedGroupKFold,  # CHANGED: stratified inner CV
                                     cross_val_predict)
from sklearn.metrics import roc_auc_score, average_precision_score

from timescoring import scoring
from timescoring.annotations import Annotation

## CONFIG

Everything you change between runs lives here. The SVM hyperparameters are
**fixed** — set them once; nothing is tuned on test data.

In [ ]:
# --- patients to run ---
# Sharding: set PATIENT_SHARD env var to run a subset across separate sessions.
#   PATIENT_SHARD="chb04"            -> just the heavy patient, its own session
#   PATIENT_SHARD="chb01,chb02,..."  -> a comma-separated list
#   unset / empty                    -> all 24
import os as _os
_ALL = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24
_shard = _os.environ.get("PATIENT_SHARD", "").strip()
if _shard:
    PATIENTS = [p.strip() for p in _shard.split(",") if p.strip()]
else:
    PATIENTS = _ALL

# --- paths ---
SEIZURE_DIR     = Path("Features/Seizure")
NON_SEIZURE_DIR = Path("Features/Non_Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- FIXED hyperparameters (no per-fold CV search) ---
# C, gamma and the feature-selection percentile are FIXED here and used as-is
# for every fold. Change these three values to run a different configuration.
cfg = {"C": 10, "gamma": 0.001, "percentile": 10}

# --- windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- operating-point calibration (threshold + persistence k) ---
THRESH_GRID       = None                       # None -> per-fold score quantiles
K_GRID            = [1, 2, 3, 4, 6, 8]         # consecutive-window persistence
FP_PER_DAY_BUDGET = 5                           # a-priori FP/day ceiling

# --- evaluation scope ---
# TEST_ON_SEIZURE_ONLY = True  -> score ONLY the seizure-containing records
#   (Loop 1). Non-seizure records are NOT tested, so event-level FP/day and
#   specificity come solely from the interictal windows inside seizure records.
# False -> also run Loop 2 (true LONO over non-seizure records), as before.
TEST_ON_SEIZURE_ONLY = False

# Inner CV used ONLY to pick the operating point (thr + k) per outer fold,
# computed on records that EXCLUDE the held-out test record -> leakage-free.
# GroupKFold keeps every record contributing to calibration (~5 fits/fold);
# set to None for full inner leave-one-record-out (clean but ~N fits/fold).
INNER_CAL_SPLITS  = 3

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# --- training-set construction (TRAINING ONLY; TEST records always scored in full) ---
# UNDERSAMPLE = False -> train on SEIZURE-CONTAINING files only, each kept IN
#                        FULL. Non-seizure files are never used for training
#                        (they are still scored as held-out TEST records,
#                        unless TEST_ON_SEIZURE_ONLY is set above).
# UNDERSAMPLE = True  -> combined-budget undersampling: each seizure record is
#                        reduced to ONE contiguous chunk (seizure span +
#                        SEIZ_MARGIN_SEC interictal padding each side) and the
#                        non-seizure records fill the remainder up to
#                        MAX_TRAIN_WINDOWS (contiguous per-record blocks).
UNDERSAMPLE       = True
MAX_TRAIN_WINDOWS = 20000     # total training windows  (used only if UNDERSAMPLE)
SEIZ_MARGIN_SEC   = 300.0     # interictal padding each side of seizure span (UNDERSAMPLE); design choice, not a fixed standard
SUBSAMPLE_SEED    = 0         # deterministic per-record seed (UNDERSAMPLE)

# --- parallelism over LORO folds ---
OUTER_N_JOBS = 8

# --- outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # combined summary across patients

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


def make_pipe(cfg):
    # Memory caching removed on purpose (it wrote a growing 'cache/' folder
    # to disk and gave no speedup). cache_size below is the in-RAM libsvm
    # kernel buffer, per worker — not disk.
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight=None, cache_size=1024)),  # CHANGED: RAM-safe across workers
    ])

## Load one patient

Returns `seizure_records` and `non_records` dicts: `{record_name: (X, y)}`.

In [ ]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Event-scoring & calibration helpers

In [ ]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n, offset_sec=0.0):
    # === CHANGED (seizure-chunk alignment): offset_sec shifts the ABSOLUTE
    # annotation times into the LOCAL coordinates of a subsampled seizure chunk
    # whose window 0 sits offset_sec into the file. Default 0.0 -> identical to
    # the old behavior for full records (test scoring, non-seizure records, and
    # chunks that were clipped to the file start).
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round((r["start_sec"] - offset_sec) * GRID_FS))
        e = int(round((r["end_sec"]   - offset_sec) * GRID_FS))
        if e <= 0 or s >= n:
            continue
        mask[max(s, 0):min(e, n)] = True
    return mask


_REF_MASK_CACHE = {}

def event_counts(record_scores, threshold, k):
    # ref_mask_from_record never depends on (threshold, k) -- only on the
    # record's ground-truth annotations and its sample-grid length n.
    # calibrate() calls this ~80-150x per patient/config with the SAME
    # record_scores (same n per record each time), so caching by
    # (record, n) avoids rebuilding an identical mask from the annotations
    # CSV every single combo. Pure speed fix -- ref mask is ground truth,
    # not a result, so caching it changes nothing about correctness.
    TP = FP = FN = 0
    # === CHANGED: each OOF record now carries a 5th item, offset_sec, used to
    # align the reference mask to a subsampled seizure chunk's local grid.
    for record, scores, y, dur, offset_sec in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        key = (record, n, offset_sec)
        ref = _REF_MASK_CACHE.get(key)
        if ref is None:
            ref = ref_mask_from_record(record, n, offset_sec)
            _REF_MASK_CACHE[key] = ref
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def calibrate(oof_record_scores):
    # Selects threshold + k (the OPERATING POINT) on out-of-fold train scores.
    # This is not model selection — the SVM is already fixed.
    # === CHANGED: OOF tuples are now 5-wide (.., offset_sec); unpack to match.
    total_days = sum(d for _, _, _, d, _ in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.50, 0.999, 20)))
    else:
        thr_grid = THRESH_GRID

    best = best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]


## LORO for one patient

Fixed SVM. Per fold: hold out one record, compute the calibration OOF on the
remaining records (inner grouped CV that EXCLUDES the held-out record), pick
(threshold, k) from those leakage-free scores, fit the deployment model on all
remaining records, and score the held-out record. Seizure records drive
sensitivity (Loop 1); seizure-free records get the same true leave-one-record-out
treatment (Loop 2) so their false alarms count.

`ap` / `roc_auc` are computed per fold on the held-out record as **diagnostics
only** — they are printed, never used to choose anything.

In [ ]:
def _stable_seed(*parts):
    """Deterministic 64-bit seed from arbitrary parts, stable ACROSS processes
    (unlike Python's hash(), which is salted per-process by PYTHONHASHSEED)."""
    s = "|".join("" if p is None else str(p) for p in parts)
    return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(),
                          "little")


def build_train(seizure_records, non_records=None, exclude=None,
                exclude_non=None):
    """Stack TRAINING data (model fits + OOF calibration only -- never test).

    Two modes, set by the UNDERSAMPLE flag in CONFIG:

    UNDERSAMPLE = False (default)
        Train on the SEIZURE-CONTAINING records ONLY, each kept IN FULL (every
        window, real-time order). Non-seizure records are not used for training
        (still scored as held-out TEST records in Loop 2). All offsets are 0.0,
        so event-level calibration aligns exactly like full records.

    UNDERSAMPLE = True
        Combined-budget undersampling. Each seizure record is reduced to ONE
        contiguous chunk spanning all its seizures (first to last label-1
        window) plus SEIZ_MARGIN_SEC of interictal padding on each side; the
        chunk's start offset (sec) is recorded so calibration can realign the
        absolute annotations. Non-seizure records then fill the remainder up to
        MAX_TRAIN_WINDOWS, each as a CONTIGUOUS block sized proportionally to
        its length (deterministic per-record seed).

    Either way TEST records are scored in FULL elsewhere -- only TRAINING is
    thinned. Returns (X, y, groups, offsets); groups are the record names.
    """
    Xs, ys, grps = [], [], []
    offsets = {}   # record_name -> seconds the kept block starts into the file

    # ---- default: seizure-containing files only, kept in full ----
    if not UNDERSAMPLE:
        for name, (X, y) in seizure_records.items():
            if name == exclude:
                continue
            Xs.append(X); ys.append(y); grps.extend([name] * len(y))
            offsets[name] = 0.0
        return np.vstack(Xs), np.concatenate(ys), np.array(grps), offsets

    # ---- UNDERSAMPLE: seizure chunks + capped non-seizure fill ----
    margin_w = int(round(SEIZ_MARGIN_SEC / STEP))
    total_seiz_kept = 0
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        pos = np.where(y == 1)[0]
        if pos.size == 0:                     # no labeled seizure -> keep full
            Xs.append(X); ys.append(y); grps.extend([name] * len(y))
            offsets[name] = 0.0
            total_seiz_kept += len(y)
            continue
        start = max(0, int(pos.min()) - margin_w)
        end   = min(len(y), int(pos.max()) + 1 + margin_w)
        Xc, yc = X[start:end], y[start:end]
        Xs.append(Xc); ys.append(yc); grps.extend([name] * len(yc))
        offsets[name] = start * STEP          # window 0 of chunk = start*STEP sec
        total_seiz_kept += len(yc)

    if non_records:
        included = [(name, X, y) for name, (X, y) in non_records.items()
                    if name != exclude_non]
        total_non = sum(len(y) for _, _, y in included)
        cap = (None if MAX_TRAIN_WINDOWS is None
               else max(0, MAX_TRAIN_WINDOWS - total_seiz_kept))
        if cap is None or total_non <= cap:
            for name, X, y in included:
                Xs.append(X); ys.append(y); grps.extend([name] * len(y))
                offsets[name] = 0.0
        elif cap == 0:
            pass                              # seizure chunks already fill budget
        else:
            for name, X, y in included:
                n_i = len(y)
                keep_i = max(1, int(round(cap * n_i / total_non)))
                if keep_i >= n_i:
                    idx = np.arange(n_i)
                else:
                    rng = np.random.default_rng(
                        _stable_seed(SUBSAMPLE_SEED, exclude, exclude_non, name))
                    start = int(rng.integers(0, n_i - keep_i + 1))
                    idx = np.arange(start, start + keep_i)   # contiguous block
                Xs.append(X[idx]); ys.append(y[idx])
                grps.extend([name] * len(idx))
                offsets[name] = 0.0

    return np.vstack(Xs), np.concatenate(ys), np.array(grps), offsets
def _inner_cv(grps):
    """=== ADDED: record-grouped, class-STRATIFIED inner CV, shared by the
    C/gamma tuning AND the thr/k OOF calibration. StratifiedGroupKFold keeps
    each whole record on one side of every split (no record's windows span
    train+val) while balancing the seizure (label-1) proportion across splits.
    Spreading the few seizure records across folds (instead of letting them
    clump, as plain GroupKFold could) also lowers the chance of an all-interictal
    training split. n_splits is capped at the number of records; deterministic
    (no shuffle). Falls back to leave-one-record-out only in the degenerate
    <2-record case."""
    n_groups = len(np.unique(grps))
    if n_groups < 2:
        return LeaveOneGroupOut()
    k = (n_groups if (INNER_CAL_SPLITS is None or n_groups <= INNER_CAL_SPLITS)
         else INNER_CAL_SPLITS)
    return StratifiedGroupKFold(n_splits=k)


def diag_scores(y, sc):
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)



def tune_cfg(Xtr, ytr, grps, cfg):
    """=== Fixed hyperparameters -- NO per-fold CV search. ===============
    C, gamma and the feature-selection percentile are taken DIRECTLY from cfg
    (set once in the CONFIG cell) and used for every fold. There is no inner
    grid search over hyperparameters anymore.

    We still compute ONE out-of-fold decision-score vector with a
    record-grouped CV, because the thr/k operating-point calibration in
    calibrate() needs OOF scores -- that is operating-point CALIBRATION, not
    model selection. Returns (cfg, oof_scores) so the callers can reuse the
    OOF for calibration.
    """
    if len(np.unique(ytr)) < 2:
        return cfg, np.zeros(len(ytr))
    cv = _inner_cv(grps)   # StratifiedGroupKFold (record-grouped + stratified)
    oof = cross_val_predict(make_pipe(cfg), Xtr, ytr, groups=grps, cv=cv,
                            method="decision_function", n_jobs=1)
    return cfg, oof
def _oof_by_record(oof_scores, ytr, grps, offsets):
    """=== ADDED: assemble per-record OOF tuples (name, scores, y, duration,
    offset_sec) from a precomputed OOF vector -- the shape calibrate() expects.
    Lets process_*_fold REUSE tune_cfg's OOF instead of recomputing it."""
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        out.append((name, oof_scores[m], ytr[m], duration_of(ytr[m]),
                    offsets.get(name, 0.0)))
    return out


def process_seizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 1 fold: hold out one seizure record, build the training pool from
    the rest (see build_train / UNDERSAMPLE), calibrate the operating point
    (thr, k) on a nested OOF computed WITHOUT this record (leakage-free), fit
    the fixed-hyperparameter model, and score the held-out record."""
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude=test_record)
    cfg, oof_scores = tune_cfg(Xtr, ytr, grps, cfg)
    thr, k = calibrate(_oof_by_record(oof_scores, ytr, grps, offsets))
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap,
                C=cfg["C"], gamma=cfg["gamma"], percentile=cfg["percentile"])  # CHANGED: tuned


def process_nonseizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 2 fold (TRUE LONO): hold out one non-seizure record and score it.
    Training uses the same pool as Loop 1 (build_train); the model is refit per
    fold (no FP leak) and (thr, k) is calibrated on a nested OOF computed
    WITHOUT this record (leakage-free)."""
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude_non=test_record)
    cfg, oof_scores = tune_cfg(Xtr, ytr, grps, cfg)
    thr, k = calibrate(_oof_by_record(oof_scores, ytr, grps, offsets))
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = non_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=float("nan"), ap=float("nan"),
                C=cfg["C"], gamma=cfg["gamma"], percentile=cfg["percentile"])  # CHANGED: tuned


## Run one patient (build tables + return summary row)

In [ ]:
def run_patient(patient, cfg):
    seizure_records, non_records = load_patient(patient)
    if not seizure_records:
        print(f"[{patient}] no seizure records — skipping")
        return None, None
    print(f"[{patient}] seizure={len(seizure_records)} "
          f"non-seizure={len(non_records)}")

    # --- LOOP 1: LORO over seizure records (sensitivity / latency) ---
    # Each fold computes its OWN calibration OOF on records that EXCLUDE the
    # held-out test record (nested) -> the operating point is leakage-free.
    seiz_results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(seizure_records, non_records, r, cfg)
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in seiz_results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        # === CHANGED: store per-fold tuned C/gamma
        meta[res["record"]] = dict(type="seizure", roc=res["roc"], ap=res["ap"],
                                   C=res["C"], gamma=res["gamma"], percentile=res["percentile"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"C={res['C']} g={res['gamma']} p={res['percentile']}  "          # CHANGED: tuned hp
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- LOOP 2: TRUE LONO over non-seizure records (specificity / FP) ---
    if non_records and not TEST_ON_SEIZURE_ONLY:
        non_results = Parallel(n_jobs=OUTER_N_JOBS)(
            delayed(process_nonseizure_fold)(seizure_records, non_records, r, cfg)
            for r in non_records.keys()
        )
        for res in non_results:
            test_outputs.append((res["record"], res["sc"], res["y"],
                                 res["dur"], res["thr"], res["k"]))
            # === CHANGED: store per-fold tuned C/gamma
            meta[res["record"]] = dict(type="nonseizure", roc=float("nan"),
                                       ap=float("nan"),
                                       C=res["C"], gamma=res["gamma"], percentile=res["percentile"])
        print(f"  [NON] {len(non_results)} records scored (true LONO)")

    # --- per-record pass: segment table + event totals ---
    TP = FP = FN = 0
    all_latencies, segment_rows = [], []
    for record, sc, y, dur, thr, k in test_outputs:
        is_seiz = meta[record]["type"] == "seizure"
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        if is_seiz:
            all_latencies.extend(detection_latencies(record, sc, thr, k))

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        if not is_seiz:
            acc = spec = sens = prec = f1 = np.nan
        segment_rows.append({
            # === CHANGED: 'config' column dropped (constant label, no info)
            "C": meta[record]["C"],
            "gamma": meta[record]["gamma"],
            "percentile": meta[record]["percentile"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })
        kind = "SEIZ" if is_seiz else "NON"
        print(f"    {record} done  ({kind})", flush=True)

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")

    # === CHANGED: C/gamma now vary per fold -> report the MODAL choice for the
    # patient (per-fold values are in the segment CSV).
    _cs = [m.get("C") for m in meta.values() if m.get("C") is not None]
    _gs = [m.get("gamma") for m in meta.values() if m.get("gamma") is not None]
    mode_C     = max(set(_cs), key=_cs.count) if _cs else None
    mode_gamma = max(set(_gs), key=_gs.count) if _gs else None
    _ps = [m.get("percentile") for m in meta.values() if m.get("percentile") is not None]
    mode_perc  = max(set(_ps), key=_ps.count) if _ps else None

    summary_row = {
        # === CHANGED: 'config' column dropped (constant label, no info)
        "C": mode_C, "gamma": mode_gamma,
        "percentile": mode_perc,
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    return summary_row, segment_rows


# Run

Fixed hyperparameters (set in the CONFIG cell). Runs every patient in `PATIENTS` (set the `PATIENT_SHARD` env var to subset).

In [ ]:
PATIENT = "chb01"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb01
[chb01] seizure=7 non-seizure=35
  [SEIZ] chb01_03  thr=-0.182 k=6  C=10 g=0.001 p=10  roc=0.981 ap=0.640  test_pos=16
  [SEIZ] chb01_04  thr=-0.118 k=6  C=10 g=0.001 p=10  roc=0.993 ap=0.681  test_pos=11
  [SEIZ] chb01_15  thr=0.032 k=6  C=10 g=0.001 p=10  roc=0.725 ap=0.065  test_pos=16
  [SEIZ] chb01_16  thr=-0.170 k=6  C=10 g=0.001 p=10  roc=0.993 ap=0.600  test_pos=21
  [SEIZ] chb01_18  thr=-0.325 k=6  C=10 g=0.001 p=10  roc=0.999 ap=0.969  test_pos=37
  [SEIZ] chb01_21  thr=-0.345 k=6  C=10 g=0.001 p=10  roc=0.995 ap=0.977  test_pos=38
  [SEIZ] chb01_26  thr=-0.510 k=6  C=10 g=0.001 p=10  roc=0.986 ap=0.933  test_pos=41
    chb01_03 done  (SEIZ)
    chb01_04 done  (SEIZ)
    chb01_15 done  (SEIZ)
    chb01_16 done  (SEIZ)
    chb01_18 done  (SEIZ)
    chb01_21 done  (SEIZ)
    chb01_26 done  (SEIZ)
  EVENT: TP=7 FP=0 FN=0  sens=1.000 prec=1.000 f1=1.000  FP/day=0.000  lat=28.0s
  >>> SAVED [chb01] -> RESULTS_SUMMARY_chb01.csv

Wrote RESUL

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb01,6.645833,7,0,0,1.0,1.0,1.0,0.0,0.0,28.0,7,0.953169,0.69495


In [ ]:
PATIENT = "chb02"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb02
[chb02] seizure=3 non-seizure=33
  [SEIZ] chb02_16+  thr=1.030 k=2  C=10 g=0.001 p=10  roc=0.656 ap=0.036  test_pos=33
  [SEIZ] chb02_16  thr=1.775 k=1  C=10 g=0.001 p=10  roc=0.718 ap=0.236  test_pos=33
  [SEIZ] chb02_19  thr=0.159 k=4  C=10 g=0.001 p=10  roc=0.952 ap=0.050  test_pos=4
    chb02_16+ done  (SEIZ)
    chb02_16 done  (SEIZ)
    chb02_19 done  (SEIZ)
  EVENT: TP=2 FP=3 FN=1  sens=0.667 prec=0.400 f1=0.500  FP/day=31.774  lat=33.0s
  >>> SAVED [chb02] -> RESULTS_SUMMARY_chb02.csv

Wrote RESULTS_SUMMARY_chb02.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2   3   1     0.666667        0.4 31.774441


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb02,2.265972,2,3,1,0.666667,0.4,0.5,31.774441,1.323935,33.0,2,0.775537,0.107425


In [ ]:
PATIENT = "chb03"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb03
[chb03] seizure=7 non-seizure=31
  [SEIZ] chb03_01  thr=-0.427 k=8  C=10 g=0.001 p=10  roc=0.893 ap=0.254  test_pos=21
  [SEIZ] chb03_02  thr=-0.369 k=8  C=10 g=0.001 p=10  roc=0.953 ap=0.241  test_pos=26
  [SEIZ] chb03_03  thr=-0.331 k=8  C=10 g=0.001 p=10  roc=0.953 ap=0.644  test_pos=28
  [SEIZ] chb03_04  thr=2.027 k=1  C=10 g=0.001 p=10  roc=0.948 ap=0.460  test_pos=21
  [SEIZ] chb03_34  thr=0.071 k=6  C=10 g=0.001 p=10  roc=0.936 ap=0.149  test_pos=19
  [SEIZ] chb03_35  thr=-0.141 k=8  C=10 g=0.001 p=10  roc=0.972 ap=0.344  test_pos=26
  [SEIZ] chb03_36  thr=1.911 k=1  C=10 g=0.001 p=10  roc=0.908 ap=0.100  test_pos=22
    chb03_01 done  (SEIZ)
    chb03_02 done  (SEIZ)
    chb03_03 done  (SEIZ)
    chb03_04 done  (SEIZ)
    chb03_34 done  (SEIZ)
    chb03_35 done  (SEIZ)
    chb03_36 done  (SEIZ)
  EVENT: TP=6 FP=4 FN=1  sens=0.857 prec=0.600 f1=0.706  FP/day=13.714  lat=51.0s
  >>> SAVED [chb03] -> RESULTS_SUMMARY_chb03.csv

Wrote RESULT

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb03,7.0,6,4,1,0.857143,0.6,0.705882,13.714286,0.571429,51.0,6,0.937441,0.313098


In [ ]:
PATIENT = "chb04"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb04
  [DROP complex] chb04_12_features.mat: 2 window(s) [np.int64(95), np.int64(135)]
[chb04] seizure=3 non-seizure=39
  [SEIZ] chb04_05  thr=2.269 k=3  C=10 g=0.001 p=10  roc=0.961 ap=0.386  test_pos=20
  [SEIZ] chb04_08  thr=0.333 k=6  C=10 g=0.001 p=10  roc=0.872 ap=0.079  test_pos=44
  [SEIZ] chb04_28  thr=-1.073 k=8  C=10 g=0.001 p=10  roc=0.849 ap=0.064  test_pos=88
    chb04_05 done  (SEIZ)
    chb04_08 done  (SEIZ)
    chb04_28 done  (SEIZ)
  EVENT: TP=2 FP=6 FN=2  sens=0.500 prec=0.250 f1=0.333  FP/day=13.515  lat=62.5s
  >>> SAVED [chb04] -> RESULTS_SUMMARY_chb04.csv

Wrote RESULTS_SUMMARY_chb04.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2   6   2          0.5       0.25 13.514958


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb04,10.654861,2,6,2,0.5,0.25,0.333333,13.514958,0.563123,62.5,2,0.893931,0.176452


In [ ]:
PATIENT = "chb05"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb05
[chb05] seizure=5 non-seizure=34
  [SEIZ] chb05_06  thr=0.326 k=6  C=10 g=0.001 p=10  roc=0.975 ap=0.743  test_pos=46
  [SEIZ] chb05_13  thr=0.267 k=8  C=10 g=0.001 p=10  roc=0.991 ap=0.792  test_pos=44
  [SEIZ] chb05_16  thr=0.202 k=6  C=10 g=0.001 p=10  roc=0.976 ap=0.743  test_pos=39
  [SEIZ] chb05_17  thr=0.188 k=4  C=10 g=0.001 p=10  roc=0.900 ap=0.474  test_pos=48
  [SEIZ] chb05_22  thr=-0.527 k=8  C=10 g=0.001 p=10  roc=0.911 ap=0.486  test_pos=47
    chb05_06 done  (SEIZ)
    chb05_13 done  (SEIZ)
    chb05_16 done  (SEIZ)
    chb05_17 done  (SEIZ)
    chb05_22 done  (SEIZ)
  EVENT: TP=5 FP=6 FN=0  sens=1.000 prec=0.455 f1=0.625  FP/day=28.800  lat=53.0s
  >>> SAVED [chb05] -> RESULTS_SUMMARY_chb05.csv

Wrote RESULTS_SUMMARY_chb05.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  5   6   0          1.0   0.454545    28.8


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb05,5.0,5,6,0,1.0,0.454545,0.625,28.8,1.2,53.0,5,0.950678,0.647527


In [ ]:
PATIENT = "chb06"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb06
[chb06] seizure=7 non-seizure=11
  [SEIZ] chb06_01  thr=0.958 k=3  C=10 g=0.001 p=10  roc=0.881 ap=0.305  test_pos=19
  [SEIZ] chb06_04  thr=0.535 k=2  C=10 g=0.001 p=10  roc=0.826 ap=0.050  test_pos=16
  [SEIZ] chb06_09  thr=1.067 k=2  C=10 g=0.001 p=10  roc=0.971 ap=0.077  test_pos=7
  [SEIZ] chb06_10  thr=1.141 k=2  C=10 g=0.001 p=10  roc=0.986 ap=0.421  test_pos=5
  [SEIZ] chb06_13  thr=0.964 k=2  C=10 g=0.001 p=10  roc=0.986 ap=0.119  test_pos=5
  [SEIZ] chb06_18  thr=0.997 k=2  C=10 g=0.001 p=10  roc=0.679 ap=0.003  test_pos=5
  [SEIZ] chb06_24  thr=1.095 k=2  C=10 g=0.001 p=10  roc=0.870 ap=0.022  test_pos=7
    chb06_01 done  (SEIZ)
    chb06_04 done  (SEIZ)
    chb06_09 done  (SEIZ)
    chb06_10 done  (SEIZ)
    chb06_13 done  (SEIZ)
    chb06_18 done  (SEIZ)
    chb06_24 done  (SEIZ)
  EVENT: TP=1 FP=0 FN=9  sens=0.100 prec=1.000 f1=0.182  FP/day=0.000  lat=12.0s
  >>> SAVED [chb06] -> RESULTS_SUMMARY_chb06.csv

Wrote RESULTS_SUMMARY_

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb06,25.892361,1,0,9,0.1,1.0,0.181818,0.0,0.0,12.0,1,0.885681,0.142277


In [ ]:
PATIENT = "chb07"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb07
[chb07] seizure=3 non-seizure=16
  [SEIZ] chb07_12  thr=1.600 k=2  C=10 g=0.001 p=10  roc=0.757 ap=0.105  test_pos=35
  [SEIZ] chb07_13  thr=1.235 k=3  C=10 g=0.001 p=10  roc=0.682 ap=0.099  test_pos=39
  [SEIZ] chb07_19  thr=0.180 k=6  C=10 g=0.001 p=10  roc=0.834 ap=0.092  test_pos=76
    chb07_12 done  (SEIZ)
    chb07_13 done  (SEIZ)
    chb07_19 done  (SEIZ)
  EVENT: TP=1 FP=1 FN=2  sens=0.333 prec=0.500 f1=0.400  FP/day=2.315  lat=15.0s
  >>> SAVED [chb07] -> RESULTS_SUMMARY_chb07.csv

Wrote RESULTS_SUMMARY_chb07.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision   FP/day
  1   1   2     0.333333        0.5 2.315423


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb07,10.365278,1,1,2,0.333333,0.5,0.4,2.315423,0.096476,15.0,1,0.757545,0.098497


In [ ]:
PATIENT = "chb08"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb08
[chb08] seizure=5 non-seizure=15
  [SEIZ] chb08_02  thr=0.584 k=4  C=10 g=0.001 p=10  roc=0.926 ap=0.604  test_pos=69
  [SEIZ] chb08_05  thr=-0.447 k=8  C=10 g=0.001 p=10  roc=0.930 ap=0.538  test_pos=76
  [SEIZ] chb08_11  thr=0.887 k=6  C=10 g=0.001 p=10  roc=0.969 ap=0.669  test_pos=53
  [SEIZ] chb08_13  thr=2.700 k=1  C=10 g=0.001 p=10  roc=0.950 ap=0.705  test_pos=64
  [SEIZ] chb08_21  thr=0.428 k=4  C=10 g=0.001 p=10  roc=0.788 ap=0.533  test_pos=105
    chb08_02 done  (SEIZ)
    chb08_05 done  (SEIZ)
    chb08_11 done  (SEIZ)
    chb08_13 done  (SEIZ)
    chb08_21 done  (SEIZ)
  EVENT: TP=4 FP=2 FN=1  sens=0.800 prec=0.667 f1=0.727  FP/day=9.600  lat=48.2s
  >>> SAVED [chb08] -> RESULTS_SUMMARY_chb08.csv

Wrote RESULTS_SUMMARY_chb08.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  4   2   1          0.8   0.666667     9.6


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb08,5.0,4,2,1,0.8,0.666667,0.727273,9.6,0.4,48.25,4,0.912399,0.609625


In [ ]:
PATIENT = "chb09"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb09
[chb09] seizure=3 non-seizure=16
  [SEIZ] chb09_06  thr=-0.621 k=6  C=10 g=0.001 p=10  roc=0.936 ap=0.551  test_pos=26
  [SEIZ] chb09_08  thr=0.797 k=2  C=10 g=0.001 p=10  roc=0.968 ap=0.745  test_pos=60
  [SEIZ] chb09_19  thr=-0.836 k=6  C=10 g=0.001 p=10  roc=0.918 ap=0.266  test_pos=25
    chb09_06 done  (SEIZ)
    chb09_08 done  (SEIZ)
    chb09_19 done  (SEIZ)
  EVENT: TP=4 FP=7 FN=0  sens=1.000 prec=0.364 f1=0.533  FP/day=17.532  lat=31.5s
  >>> SAVED [chb09] -> RESULTS_SUMMARY_chb09.csv

Wrote RESULTS_SUMMARY_chb09.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  4   7   0          1.0   0.363636 17.531705


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb09,9.582639,4,7,0,1.0,0.363636,0.533333,17.531705,0.730488,31.5,4,0.940663,0.520627


In [ ]:
PATIENT = "chb10"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb10
[chb10] seizure=7 non-seizure=18
  [SEIZ] chb10_12  thr=-0.404 k=6  C=10 g=0.001 p=10  roc=0.982 ap=0.438  test_pos=14
  [SEIZ] chb10_20  thr=-0.818 k=8  C=10 g=0.001 p=10  roc=0.879 ap=0.533  test_pos=28
  [SEIZ] chb10_27  thr=-0.479 k=6  C=10 g=0.001 p=10  roc=1.000 ap=0.985  test_pos=26
  [SEIZ] chb10_30  thr=-0.201 k=6  C=10 g=0.001 p=10  roc=0.994 ap=0.737  test_pos=23
  [SEIZ] chb10_31  thr=-0.464 k=6  C=10 g=0.001 p=10  roc=0.993 ap=0.927  test_pos=30
  [SEIZ] chb10_38  thr=-0.528 k=6  C=10 g=0.001 p=10  roc=0.932 ap=0.677  test_pos=35
  [SEIZ] chb10_89  thr=-0.501 k=6  C=10 g=0.001 p=10  roc=0.919 ap=0.458  test_pos=21
    chb10_12 done  (SEIZ)
    chb10_20 done  (SEIZ)
    chb10_27 done  (SEIZ)
    chb10_30 done  (SEIZ)
    chb10_31 done  (SEIZ)
    chb10_38 done  (SEIZ)
    chb10_89 done  (SEIZ)
  EVENT: TP=6 FP=2 FN=1  sens=0.857 prec=0.750 f1=0.800  FP/day=3.424  lat=18.5s
  >>> SAVED [chb10] -> RESULTS_SUMMARY_chb10.csv

Wrote RESU

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb10,14.016667,6,2,1,0.857143,0.75,0.8,3.424495,0.142687,18.5,6,0.95703,0.67908


In [ ]:
PATIENT = "chb11"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb11
[chb11] seizure=3 non-seizure=32
  [SEIZ] chb11_82  thr=-0.808 k=8  C=10 g=0.001 p=10  roc=0.998 ap=0.635  test_pos=9
  [SEIZ] chb11_92  thr=-0.436 k=6  C=10 g=0.001 p=10  roc=0.666 ap=0.021  test_pos=13
  [SEIZ] chb11_99  thr=0.047 k=2  C=10 g=0.001 p=10  roc=0.808 ap=0.653  test_pos=301
    chb11_82 done  (SEIZ)
    chb11_92 done  (SEIZ)
    chb11_99 done  (SEIZ)
  EVENT: TP=3 FP=1 FN=2  sens=0.600 prec=0.750 f1=0.667  FP/day=8.591  lat=2.8s
  >>> SAVED [chb11] -> RESULTS_SUMMARY_chb11.csv

Wrote RESULTS_SUMMARY_chb11.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision   FP/day
  3   1   2          0.6       0.75 8.590604


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb11,2.79375,3,1,2,0.6,0.75,0.666667,8.590604,0.357942,2.75,2,0.823967,0.436676


In [ ]:
PATIENT = "chb12"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb12
[chb12] seizure=10 non-seizure=11
  [SEIZ] chb12_06  thr=2.194 k=1  C=10 g=0.001 p=10  roc=0.762 ap=0.276  test_pos=38
  [SEIZ] chb12_08  thr=-0.431 k=8  C=10 g=0.001 p=10  roc=0.843 ap=0.521  test_pos=32
  [SEIZ] chb12_09  thr=2.344 k=1  C=10 g=0.001 p=10  roc=0.934 ap=0.675  test_pos=26
  [SEIZ] chb12_10  thr=1.991 k=1  C=10 g=0.001 p=10  roc=0.962 ap=0.751  test_pos=31
  [SEIZ] chb12_11  thr=2.195 k=1  C=10 g=0.001 p=10  roc=0.884 ap=0.330  test_pos=15
  [SEIZ] chb12_23  thr=2.430 k=1  C=10 g=0.001 p=10  roc=0.731 ap=0.270  test_pos=88
  [SEIZ] chb12_33  thr=0.166 k=6  C=10 g=0.001 p=10  roc=0.795 ap=0.148  test_pos=19
  [SEIZ] chb12_36  thr=0.196 k=8  C=10 g=0.001 p=10  roc=0.921 ap=0.135  test_pos=11
  [SEIZ] chb12_38  thr=2.282 k=1  C=10 g=0.001 p=10  roc=0.848 ap=0.469  test_pos=77
  [SEIZ] chb12_42  thr=2.223 k=1  C=10 g=0.001 p=10  roc=0.830 ap=0.483  test_pos=64
    chb12_06 done  (SEIZ)
    chb12_08 done  (SEIZ)
    chb12_09 done  (S

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb12,9.681944,12,0,15,0.444444,1.0,0.615385,0.0,0.0,29.0,12,0.851052,0.405791


In [ ]:
PATIENT = "chb13"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb13
[chb13] seizure=8 non-seizure=25
  [SEIZ] chb13_19  thr=1.726 k=2  C=10 g=0.001 p=10  roc=0.580 ap=0.032  test_pos=18
  [SEIZ] chb13_21  thr=1.149 k=2  C=10 g=0.001 p=10  roc=0.656 ap=0.100  test_pos=28
  [SEIZ] chb13_40  thr=2.116 k=2  C=10 g=0.001 p=10  roc=0.561 ap=0.034  test_pos=39
  [SEIZ] chb13_55  thr=1.179 k=2  C=10 g=0.001 p=10  roc=0.710 ap=0.030  test_pos=15
  [SEIZ] chb13_58  thr=1.566 k=2  C=10 g=0.001 p=10  roc=0.832 ap=0.034  test_pos=7
  [SEIZ] chb13_59  thr=-0.569 k=8  C=10 g=0.001 p=10  roc=0.696 ap=0.056  test_pos=25
  [SEIZ] chb13_60  thr=1.257 k=2  C=10 g=0.001 p=10  roc=0.758 ap=0.022  test_pos=9
  [SEIZ] chb13_62  thr=1.874 k=3  C=10 g=0.001 p=10  roc=0.579 ap=0.079  test_pos=75
    chb13_19 done  (SEIZ)
    chb13_21 done  (SEIZ)
    chb13_40 done  (SEIZ)
    chb13_55 done  (SEIZ)
    chb13_58 done  (SEIZ)
    chb13_59 done  (SEIZ)
    chb13_60 done  (SEIZ)
    chb13_62 done  (SEIZ)
  EVENT: TP=0 FP=0 FN=12  sens=0.000 p

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb13,8.0,0,0,12,0.0,0.0,0.0,0.0,0.0,NaN,0,0.671429,0.048163


In [ ]:
PATIENT = "chb14"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb14
[chb14] seizure=7 non-seizure=19
  [SEIZ] chb14_03  thr=0.249 k=2  C=10 g=0.001 p=10  roc=0.997 ap=0.711  test_pos=6
  [SEIZ] chb14_04  thr=0.114 k=2  C=10 g=0.001 p=10  roc=0.958 ap=0.365  test_pos=17
  [SEIZ] chb14_06  thr=-1.086 k=6  C=10 g=0.001 p=10  roc=0.971 ap=0.391  test_pos=6
  [SEIZ] chb14_11  thr=-1.204 k=6  C=10 g=0.001 p=10  roc=0.906 ap=0.429  test_pos=16
  [SEIZ] chb14_17  thr=-0.741 k=6  C=10 g=0.001 p=10  roc=0.984 ap=0.279  test_pos=8
  [SEIZ] chb14_18  thr=0.360 k=2  C=10 g=0.001 p=10  roc=0.935 ap=0.393  test_pos=9
  [SEIZ] chb14_27  thr=-0.833 k=6  C=10 g=0.001 p=10  roc=0.984 ap=0.265  test_pos=6
    chb14_03 done  (SEIZ)
    chb14_04 done  (SEIZ)
    chb14_06 done  (SEIZ)
    chb14_11 done  (SEIZ)
    chb14_17 done  (SEIZ)
    chb14_18 done  (SEIZ)
    chb14_27 done  (SEIZ)
  EVENT: TP=3 FP=3 FN=5  sens=0.375 prec=0.500 f1=0.429  FP/day=10.286  lat=13.0s
  >>> SAVED [chb14] -> RESULTS_SUMMARY_chb14.csv

Wrote RESULTS_SUM

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb14,7.0,3,3,5,0.375,0.5,0.428571,10.285714,0.428571,13.0,3,0.962009,0.40474


In [ ]:
PATIENT = "chb15"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb15
[chb15] seizure=14 non-seizure=26
  [SEIZ] chb15_06  thr=-0.002 k=8  C=10 g=0.001 p=10  roc=0.989 ap=0.805  test_pos=50
  [SEIZ] chb15_10  thr=0.828 k=8  C=10 g=0.001 p=10  roc=0.996 ap=0.714  test_pos=13
  [SEIZ] chb15_15  thr=0.697 k=8  C=10 g=0.001 p=10  roc=0.882 ap=0.521  test_pos=63
  [SEIZ] chb15_17  thr=-0.287 k=8  C=10 g=0.001 p=10  roc=0.804 ap=0.064  test_pos=15
  [SEIZ] chb15_20  thr=0.521 k=6  C=10 g=0.001 p=10  roc=0.796 ap=0.061  test_pos=22
  [SEIZ] chb15_22  thr=0.563 k=8  C=10 g=0.001 p=10  roc=0.846 ap=0.213  test_pos=83
  [SEIZ] chb15_28  thr=-0.363 k=8  C=10 g=0.001 p=10  roc=0.926 ap=0.430  test_pos=76
  [SEIZ] chb15_31  thr=-0.017 k=8  C=10 g=0.001 p=10  roc=0.942 ap=0.615  test_pos=48
  [SEIZ] chb15_40  thr=0.640 k=8  C=10 g=0.001 p=10  roc=0.901 ap=0.658  test_pos=97
  [SEIZ] chb15_46  thr=0.050 k=8  C=10 g=0.001 p=10  roc=0.750 ap=0.498  test_pos=43
  [SEIZ] chb15_49  thr=0.737 k=8  C=10 g=0.001 p=10  roc=0.882 ap=0.50

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb15,14.009028,9,4,11,0.45,0.692308,0.545455,6.852724,0.28553,26.5,9,0.898995,0.492034


In [ ]:
PATIENT = "chb16"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb16
[chb16] seizure=6 non-seizure=13
  [SEIZ] chb16_10  thr=1.309 k=2  C=10 g=0.001 p=10  roc=0.720 ap=0.010  test_pos=4
  [SEIZ] chb16_11  thr=0.785 k=2  C=10 g=0.001 p=10  roc=0.899 ap=0.072  test_pos=4
  [SEIZ] chb16_14  thr=0.368 k=2  C=10 g=0.001 p=10  roc=0.312 ap=0.003  test_pos=6
  [SEIZ] chb16_16  thr=0.738 k=2  C=10 g=0.001 p=10  roc=0.804 ap=0.011  test_pos=3
  [SEIZ] chb16_17  thr=0.524 k=3  C=10 g=0.001 p=10  roc=0.589 ap=0.017  test_pos=15
  [SEIZ] chb16_18  thr=0.549 k=2  C=10 g=0.001 p=10  roc=0.816 ap=0.086  test_pos=7
    chb16_10 done  (SEIZ)
    chb16_11 done  (SEIZ)
    chb16_14 done  (SEIZ)
    chb16_16 done  (SEIZ)
    chb16_17 done  (SEIZ)
    chb16_18 done  (SEIZ)
  EVENT: TP=1 FP=4 FN=9  sens=0.100 prec=0.200 f1=0.133  FP/day=16.000  lat=11.0s
  >>> SAVED [chb16] -> RESULTS_SUMMARY_chb16.csv

Wrote RESULTS_SUMMARY_chb16.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  1   4   9          0.1        0.2

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb16,6.0,1,4,9,0.1,0.2,0.133333,16.0,0.666667,11.0,1,0.689854,0.033131


In [ ]:
PATIENT = "chb17"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb17
[chb17] seizure=3 non-seizure=18
  [SEIZ] chb17a_03  thr=2.035 k=1  C=10 g=0.001 p=10  roc=0.828 ap=0.221  test_pos=36
  [SEIZ] chb17a_04  thr=0.342 k=1  C=10 g=0.001 p=10  roc=0.856 ap=0.281  test_pos=47
  [SEIZ] chb17b_63  thr=1.685 k=2  C=10 g=0.001 p=10  roc=0.974 ap=0.727  test_pos=35
    chb17a_03 done  (SEIZ)
    chb17a_04 done  (SEIZ)
    chb17b_63 done  (SEIZ)
  EVENT: TP=3 FP=5 FN=0  sens=1.000 prec=0.375 f1=0.545  FP/day=39.917  lat=36.5s
  >>> SAVED [chb17] -> RESULTS_SUMMARY_chb17.csv

Wrote RESULTS_SUMMARY_chb17.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision   FP/day
  3   5   0          1.0      0.375 39.91684


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb17,3.00625,3,5,0,1.0,0.375,0.545455,39.91684,1.663202,36.5,3,0.886365,0.409518


In [ ]:
PATIENT = "chb18"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb18
[chb18] seizure=6 non-seizure=30
  [SEIZ] chb18_29  thr=1.633 k=1  C=10 g=0.001 p=10  roc=0.791 ap=0.100  test_pos=20
  [SEIZ] chb18_30  thr=2.127 k=1  C=10 g=0.001 p=10  roc=0.464 ap=0.008  test_pos=12
  [SEIZ] chb18_31  thr=1.669 k=1  C=10 g=0.001 p=10  roc=0.918 ap=0.654  test_pos=28
  [SEIZ] chb18_32  thr=0.937 k=2  C=10 g=0.001 p=10  roc=0.967 ap=0.325  test_pos=22
  [SEIZ] chb18_35  thr=-0.089 k=6  C=10 g=0.001 p=10  roc=0.805 ap=0.338  test_pos=27
  [SEIZ] chb18_36  thr=-0.058 k=8  C=10 g=0.001 p=10  roc=0.832 ap=0.417  test_pos=18
    chb18_29 done  (SEIZ)
    chb18_30 done  (SEIZ)
    chb18_31 done  (SEIZ)
    chb18_32 done  (SEIZ)
    chb18_35 done  (SEIZ)
    chb18_36 done  (SEIZ)
  EVENT: TP=2 FP=4 FN=4  sens=0.333 prec=0.333 f1=0.333  FP/day=17.048  lat=34.8s
  >>> SAVED [chb18] -> RESULTS_SUMMARY_chb18.csv

Wrote RESULTS_SUMMARY_chb18.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2   4   4     0.333333  

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb18,5.63125,2,4,4,0.333333,0.333333,0.333333,17.047725,0.710322,34.75,2,0.796281,0.306997


In [ ]:
PATIENT = "chb19"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb19
[chb19] seizure=3 non-seizure=27
  [SEIZ] chb19_28  thr=1.675 k=1  C=10 g=0.001 p=10  roc=0.829 ap=0.192  test_pos=31
  [SEIZ] chb19_29  thr=2.071 k=3  C=10 g=0.001 p=10  roc=0.960 ap=0.399  test_pos=31
  [SEIZ] chb19_30  thr=1.645 k=1  C=10 g=0.001 p=10  roc=0.704 ap=0.066  test_pos=33
    chb19_28 done  (SEIZ)
    chb19_29 done  (SEIZ)
    chb19_30 done  (SEIZ)
  EVENT: TP=2 FP=9 FN=1  sens=0.667 prec=0.182 f1=0.286  FP/day=73.759  lat=21.0s
  >>> SAVED [chb19] -> RESULTS_SUMMARY_chb19.csv

Wrote RESULTS_SUMMARY_chb19.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision    FP/day
  2   9   1     0.666667   0.181818 73.758596


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb19,2.928472,2,9,1,0.666667,0.181818,0.285714,73.758596,3.073275,21.0,2,0.830862,0.219051


In [ ]:
PATIENT = "chb20"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb20
[chb20] seizure=6 non-seizure=23
  [SEIZ] chb20_12  thr=-0.182 k=8  C=10 g=0.001 p=10  roc=0.992 ap=0.503  test_pos=12
  [SEIZ] chb20_13  thr=1.734 k=3  C=10 g=0.001 p=10  roc=0.985 ap=0.654  test_pos=28
  [SEIZ] chb20_14  thr=1.411 k=3  C=10 g=0.001 p=10  roc=0.993 ap=0.491  test_pos=15
  [SEIZ] chb20_15  thr=1.311 k=4  C=10 g=0.001 p=10  roc=0.979 ap=0.727  test_pos=35
  [SEIZ] chb20_16  thr=1.575 k=3  C=10 g=0.001 p=10  roc=0.890 ap=0.134  test_pos=14
  [SEIZ] chb20_68  thr=-0.256 k=8  C=10 g=0.001 p=10  roc=0.964 ap=0.204  test_pos=15
    chb20_12 done  (SEIZ)
    chb20_13 done  (SEIZ)
    chb20_14 done  (SEIZ)
    chb20_15 done  (SEIZ)
    chb20_16 done  (SEIZ)
    chb20_68 done  (SEIZ)
  EVENT: TP=2 FP=0 FN=6  sens=0.250 prec=1.000 f1=0.400  FP/day=0.000  lat=31.5s
  >>> SAVED [chb20] -> RESULTS_SUMMARY_chb20.csv

Wrote RESULTS_SUMMARY_chb20.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  2   0   6         0.25     

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb20,5.564583,2,0,6,0.25,1.0,0.4,0.0,0.0,31.5,2,0.967176,0.452418


In [ ]:
PATIENT = "chb21"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb21
[chb21] seizure=4 non-seizure=29
  [SEIZ] chb21_19  thr=-0.955 k=6  C=10 g=0.001 p=10  roc=0.981 ap=0.782  test_pos=22
  [SEIZ] chb21_20  thr=1.084 k=1  C=10 g=0.001 p=10  roc=0.917 ap=0.439  test_pos=20
  [SEIZ] chb21_21  thr=0.685 k=1  C=10 g=0.001 p=10  roc=0.980 ap=0.654  test_pos=32
  [SEIZ] chb21_22  thr=-0.823 k=6  C=10 g=0.001 p=10  roc=0.998 ap=0.686  test_pos=5
    chb21_19 done  (SEIZ)
    chb21_20 done  (SEIZ)
    chb21_21 done  (SEIZ)
    chb21_22 done  (SEIZ)
  EVENT: TP=4 FP=1 FN=0  sens=1.000 prec=0.800 f1=0.889  FP/day=6.265  lat=13.2s
  >>> SAVED [chb21] -> RESULTS_SUMMARY_chb21.csv

Wrote RESULTS_SUMMARY_chb21.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  4   1   0          1.0        0.8 6.26541


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb21,3.830556,4,1,0,1.0,0.8,0.888889,6.26541,0.261059,13.25,4,0.968867,0.640223


In [ ]:
PATIENT = "chb22"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb22
  [DROP complex] chb22_11_features.mat: 1 window(s) [np.int64(1206)]
[chb22] seizure=3 non-seizure=28
  [SEIZ] chb22_20  thr=0.901 k=1  C=10 g=0.001 p=10  roc=0.988 ap=0.792  test_pos=24
  [SEIZ] chb22_25  thr=1.027 k=1  C=10 g=0.001 p=10  roc=0.973 ap=0.704  test_pos=30
  [SEIZ] chb22_38  thr=1.144 k=1  C=10 g=0.001 p=10  roc=0.999 ap=0.958  test_pos=29
    chb22_20 done  (SEIZ)
    chb22_25 done  (SEIZ)
    chb22_38 done  (SEIZ)
  EVENT: TP=3 FP=1 FN=0  sens=1.000 prec=0.750 f1=0.857  FP/day=8.000  lat=36.0s
  >>> SAVED [chb22] -> RESULTS_SUMMARY_chb22.csv

Wrote RESULTS_SUMMARY_chb22.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  3   1   0          1.0       0.75     8.0


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb22,3.0,3,1,0,1.0,0.75,0.857143,8.0,0.333333,36.0,3,0.986658,0.818325


In [ ]:
PATIENT = "chb23"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb23
[chb23] seizure=3 non-seizure=6
  [SEIZ] chb23_06  thr=1.665 k=1  C=10 g=0.001 p=10  roc=0.986 ap=0.818  test_pos=46
  [SEIZ] chb23_08  thr=-0.541 k=6  C=10 g=0.001 p=10  roc=0.980 ap=0.410  test_pos=28
  [SEIZ] chb23_09  thr=1.463 k=1  C=10 g=0.001 p=10  roc=0.947 ap=0.538  test_pos=99
    chb23_06 done  (SEIZ)
    chb23_08 done  (SEIZ)
    chb23_09 done  (SEIZ)
  EVENT: TP=4 FP=2 FN=3  sens=0.571 prec=0.667 f1=0.615  FP/day=5.358  lat=20.8s
  >>> SAVED [chb23] -> RESULTS_SUMMARY_chb23.csv

Wrote RESULTS_SUMMARY_chb23.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision  FP/day
  4   2   3     0.571429   0.666667 5.35814


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb23,8.958333,4,2,3,0.571429,0.666667,0.615385,5.35814,0.223256,20.75,4,0.970898,0.588815


In [ ]:
PATIENT = "chb24"          # <-- the single patient to run

print(f"fixed hyperparameters  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments = [], []

for patient in [PATIENT]:
    try:
        srow, segrows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


fixed hyperparameters  |  patient = chb24
[chb24] seizure=12 non-seizure=10
  [SEIZ] chb24_01  thr=-0.761 k=8  C=10 g=0.001 p=10  roc=0.989 ap=0.314  test_pos=11
  [SEIZ] chb24_03  thr=-0.764 k=8  C=10 g=0.001 p=10  roc=0.996 ap=0.675  test_pos=12
  [SEIZ] chb24_04  thr=0.975 k=3  C=10 g=0.001 p=10  roc=0.917 ap=0.202  test_pos=13
  [SEIZ] chb24_06  thr=0.436 k=3  C=10 g=0.001 p=10  roc=0.859 ap=0.441  test_pos=10
  [SEIZ] chb24_07  thr=0.600 k=3  C=10 g=0.001 p=10  roc=0.839 ap=0.085  test_pos=9
  [SEIZ] chb24_09  thr=0.744 k=2  C=10 g=0.001 p=10  roc=0.976 ap=0.239  test_pos=8
  [SEIZ] chb24_11  thr=0.648 k=2  C=10 g=0.001 p=10  roc=0.551 ap=0.113  test_pos=28
  [SEIZ] chb24_13  thr=0.886 k=3  C=10 g=0.001 p=10  roc=0.892 ap=0.182  test_pos=6
  [SEIZ] chb24_14  thr=0.571 k=3  C=10 g=0.001 p=10  roc=0.947 ap=0.121  test_pos=11
  [SEIZ] chb24_15  thr=0.627 k=2  C=10 g=0.001 p=10  roc=0.860 ap=0.592  test_pos=7
  [SEIZ] chb24_17  thr=-0.756 k=8  C=10 g=0.001 p=10  roc=0.903 ap=0.287  te

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb24,12.0,2,2,10,0.166667,0.5,0.25,4.0,0.166667,23.25,2,0.885195,0.299638
